# Clase 05: Análisis de valores atípicos (Outliers)

Este notebook aborda el estudio de **valores atípicos** (outliers), puntos de datos que se desvían significativamente del resto de las observaciones. La detección de outliers es crucial en minería de datos ya que pueden:
1. Indicar errores en la recolección de datos.
2. Revelar eventos raros o anomalías interesantes.
3. Sesgar significativamente los modelos estadísticos y de machine learning.

En esta notebook, veremos técnicas tanto univariadas como multivariadas utilizando implementaciones manuales y librerías estándar como `scikit-learn`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from scipy.stats import zscore
from sklearn.neighbors import LocalOutlierFactor
from sklearn.ensemble import IsolationForest
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

%matplotlib inline
sns.set_theme(style="whitegrid")

## 1. Generación de Datos de Ejemplo

Para ilustrar los métodos, generaremos un dataset sintético en 2D que contiene dos "clusters" de datos normales (inliers) y un conjunto de puntos distribuidos aleatoriamente que actuarán como outliers.

In [ ]:
# Semilla aleatoria para reproductibilidad
rng = np.random.RandomState(42)

# Generamos puntos con distribuciones "normales"
X_normales = 0.3 * rng.randn(200, 2)
# Armamos 2 grupos (clusters) normales con los datos bien separados
X_normales = np.concatenate([X_normales + 1, X_normales - 1])

# Generamos algunos outliers
X_outliers = rng.uniform(low=-7, high=7, size=(40, 2))
X = np.concatenate([X_normales, X_outliers])

# Etiquetas (1 para inliers, -1 para outliers)
n_outliers = len(X_outliers)
ground_truth = np.ones(len(X), dtype=int)
ground_truth[-n_outliers:] = -1

print(f"Total de puntos: {len(X)}")
print(f"Cantidad de outliers reales: {n_outliers}")

### Visualización del Dataset

Graficamos los datos para observar la distribución espacial. Los puntos en el centro de los clusters son los "normales", mientras que los dispersos son nuestros candidatos a outliers.

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(X[:, 0], X[:, 1], c="white", s=30, edgecolor="k", label="Datos")
plt.title("Distribución de datos con outliers sintéticos")
plt.xlabel("Característica 1")
plt.ylabel("Característica 2")
plt.legend()
plt.show()

## 2. Métodos Univariados

Los métodos univariados analizan cada variable de forma independiente. Son útiles como primer paso exploratorio para identificar valores extremos en dimensiones específicas.

### 2.1. Herramientas Visuales: Boxplots e Histogramas

El **boxplot** es una herramienta excelente para detectar outliers basados en la distribución de cuartiles. Los puntos que caen más allá de los "bigotes" son considerados atípicos.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.boxplot(X[:, 0])
ax1.set_title("Boxplot de la Característica 1")

ax2.hist(X[:, 0], bins=40, edgecolor='black')
ax2.set_title("Histograma de la Característica 1")

plt.show()

### 2.2. Método del Rango Intercuartílico (IQR)

Este método estadístico define como outliers a aquellos puntos que se encuentran fuera del rango:
$$[Q1 - 1.5 \cdot IQR, Q3 + 1.5 \cdot IQR]$$
Donde $IQR = Q3 - Q1$.

In [ ]:
def detect_atipico_iqr(x):
    q1, q3 = np.percentile(x, [25, 75])
    iqr = q3 - q1
    lim_inf = q1 - (iqr * 1.5)
    lim_sup = q3 + (iqr * 1.5)
    return (x < lim_inf) | (x > lim_sup)

outliers_iqr_c1 = detect_atipico_iqr(X[:, 0])
print(f"Outliers detectados por IQR en Columna 1: {outliers_iqr_c1.sum()}")

In [ ]:
# Visualización de outliers detectados por IQR (Univariado en C1)
plt.figure(figsize=(8, 6))
plt.scatter(X[~outliers_iqr_c1, 0], X[~outliers_iqr_c1, 1], c="white", s=30, edgecolor="k", label="Normales")
plt.scatter(X[outliers_iqr_c1, 0], X[outliers_iqr_c1, 1], c="red", s=50, edgecolor="k", label="Atípicos (IQR C1)")
plt.title("Detección Univariada con IQR sobre Característica 1")
plt.legend()
plt.show()

### 2.3. Z-Score (Puntaje Estándar)

El **Z-Score** mide cuántas desviaciones estándar se aleja un punto de la media. Se asume que los datos siguen una distribución normal. Un umbral común es $|Z| > 3$.

$$Z = \frac{x - \mu}{\sigma}$$

**Nota Importante:** En distribuciones multimodales (como la nuestra, que tiene dos clusters separados), la desviación estándar global ($\\sigma$) aumenta significativamente. Esto hace que puntos que están visualmente lejos de sus clusters no alcancen el umbral de 3 desviaciones estándar respecto a la media global.

In [ ]:
z_scores = zscore(X[:, 0])
outliers_z = np.abs(z_scores) > 3
print(f"Outliers detectados por Z-Score (>3) en Columna 1: {outliers_z.sum()}")

## 3. Métodos Multivariados

A menudo, un punto no es un outlier en ninguna de sus dimensiones por separado, pero sí lo es cuando se consideran de manera conjunta.

### 3.1. Preparación para Métodos Multivariados

Muchos métodos multivariados se basan en **distancias** (Euclídea, Mahalanobis, etc.). Si las variables tienen escalas muy diferentes (por ejemplo, edad en años vs salario en miles), la variable con mayor magnitud dominará el cálculo de la distancia, sesgando los resultados.

Por ello, es fundamental **estandarizar** o **normalizar** los datos antes de aplicar muchos de estos algoritmos.

In [ ]:
# Estandarizamos los datos para que tengan media 0 y varianza 1
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Media de los datos originales: {np.mean(X, axis=0).round(4)}")
print(f"Desviación estándar de los datos originales: {np.std(X, axis=0)}")
print(f"Media de los datos escalados: {np.mean(X_scaled, axis=0).round(4)}")
print(f"Desviación estándar de los datos escalados: {np.std(X_scaled, axis=0)}")

### 3.2. Distancia de Mahalanobis

Mide la distancia de un punto al centro de la distribución considerando la correlación entre las variables mediante la matriz de covarianza.

In [ ]:
# Porque funciona esto lo van a ver en AID, aca lo usamos como herramienta
def mahalanobis(data):
    # Centra los datos
    x_minus_mu = data - np.mean(data, axis=0)
    # Arma matríz de cov
    cov = np.cov(data.T)
    # Calculamos la inversa de la matriz de covarianza
    inv_cov = np.linalg.inv(cov)
    # Cálculo eficiente de (x-mu)S^-1(x-mu)^T para cada fila
    left_term = np.dot(x_minus_mu, inv_cov)
    mahal = np.sum(left_term * x_minus_mu, axis=1)
    return mahal

# Aplicamos Mahalanobis sobre los datos escalados
dist_m = mahalanobis(X_scaled)

top_outliers_idx = np.argsort(dist_m)[-5:]
print(f"Índices de mayores outliers (Mahalanobis): {top_outliers_idx}")
print(f"Los datos outliers son: {X[top_outliers_idx]}")

Podemos visualizar los datos detectados:

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(X[:, 0], X[:, 1], c="white", s=30, edgecolor="k", label="Datos")

# outliers
plt.scatter(X[top_outliers_idx, 0],
            X[top_outliers_idx, 1],
            c="red", s=50, edgecolor="k", label="Outliers")

plt.title("Distribución de datos con outliers sintéticos (outliers según Mahalanobis marcados)")
plt.xlabel("Característica 1")
plt.ylabel("Característica 2")
plt.legend()
plt.show()

### 3.3. Local Outlier Factor (LOF)

LOF es sensible a la densidad local. Al igual que otros métodos basados en vecinos, se beneficia enormemente de la normalización previa.

In [ ]:
# Usamos X_scaled para que las distancias sean comparables entre ejes
clf_lof = LocalOutlierFactor(n_neighbors=20, contamination=0.1)
y_pred_lof = clf_lof.fit_predict(X_scaled)

n_errors = (y_pred_lof != ground_truth).sum()
print(f"Errores de predicción LOF: {n_errors}")

X_scores = -clf_lof.negative_outlier_factor_

In [ ]:
plt.figure(figsize=(8, 6))
plt.title("Local Outlier Factor (LOF) sobre datos escalados")
plt.scatter(X_scaled[:, 0], X_scaled[:, 1], color="k", s=3.0, label="Puntos")
radius = (X_scores - X_scores.min()) / (X_scores.max() - X_scores.min())
plt.scatter(X_scaled[:, 0], X_scaled[:, 1], s=1000 * radius, edgecolors="r", facecolors="none")
plt.show()

### 3.4. Isolation Forest

Aunque los árboles de decisión (base de Isolation Forest) son robustos a las escalas de los datos, en pipelines de minería de datos solemos mantener la consistencia del escalado.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_scaled, ground_truth, test_size=0.2, random_state=42)

clf_if = IsolationForest(contamination=0.1, random_state=42)
clf_if.fit(X_train)

y_pred_if = clf_if.predict(X_test)
print(f"Outliers detectados en Test: {(y_pred_if == -1).sum()} de {(y_test == -1).sum()} reales")

In [ ]:
# Graficamos las fronteras de decisión
xx, yy = np.meshgrid(np.linspace(-5, 5, 50), np.linspace(-5, 5, 50))
# Usamos column_stack para unir las coordenadas de la malla
Z = clf_if.decision_function(np.column_stack([xx.ravel(), yy.ravel()]))
Z = Z.reshape(xx.shape)

plt.figure(figsize=(8, 6))
plt.title("Fronteras de Decisión: Isolation Forest")
plt.contourf(xx, yy, Z, cmap=plt.cm.Blues_r)

plt.scatter(X_train[:, 0], X_train[:, 1], c="white", s=20, edgecolor="k", label="Entrenamiento")
plt.scatter(X_test[:, 0], X_test[:, 1], c="green", s=20, edgecolor="k", label="Test")
plt.legend()
plt.show()